# HW 2 — Blackjack

Using Q learning, training agent to learn when to hit or stay.

A.) Environment

- Goal: The Goal is to get as close to 21 as possible while not going over 21. Also make sure to get a higher number than the dealer.
- State: (player total, dealer's visible card, usable ace)
- Actions: 0 = stay and 1 = hit.
- Transitions: Cards are continuously drawn for every hit until player stays (usually above 16)
- Rewards: win += 1, lose -= 1, and ties leads to no points.
- Termination of game: When player decides to stay with their current hand or busts due to getting over 21. It will go to dealers turn afterwards and compare # and give rewards.
- Settings: `natural=False`, `sab=False`.



In [ ]:
import os
import random
from collections import defaultdict

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython.display import Video, display

os.makedirs("results", exist_ok=True)
EPISODES = 50000

## B. Q-learning

The Q-table stores two values for every state: the value of sticking and the value of hitting. Both start at zero.

- **Alpha (α):** learning rate — how much new experience changes a Q-value.
- **Gamma (γ):** discount factor — how much future rewards matter.
- **Epsilon (ε):** exploration probability — how often the agent chooses a random action.

The update is `Q(s,a) = Q(s,a) + alpha * (reward + gamma * best_next_Q - Q(s,a))`. If a game ended, `best_next_Q = 0`.

In [ ]:
def best_action(q, state):
    # The actions are 0 = stick and 1 = hit.
    if q[state][1] > q[state][0]:
        return 1
    return 0


def train(alpha, gamma, epsilon):
    env = gym.make("Blackjack-v1", natural=False, sab=False)
    env.reset(seed=42)
    random.seed(42)
    q = defaultdict(lambda: [0.0, 0.0])
    rewards = []

    for episode in range(EPISODES):
        state, _ = env.reset()
        done = False

        while not done:
            # Explore sometimes; otherwise use the best-known action.
            if random.random() < epsilon:
                action = random.choice([0, 1])
            else:
                action = best_action(q, state)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # A finished game has no future reward.
            if done:
                future = 0
            else:
                future = max(q[next_state])

            q[state][action] += alpha * (reward + gamma * future - q[state][action])
            state = next_state

        rewards.append(reward)

    env.close()
    return q, rewards

## C–F. Run the experiments

For a fair comparison, only **one parameter changes at a time**. The baseline is `alpha=0.1, gamma=0.9, epsilon=0.2`, with **50,000 episodes** and constant epsilon (no decay).

- **C:** Baseline training.
- **D:** Learning rates 0.1, 0.5, 0.9.
- **E:** Discount factors 0.5, 0.9, 0.99.
- **F:** Exploration rates 0.05, 0.2, 0.5.

Seven training runs are enough because the baseline is shared across all three comparisons.

In [ ]:
tests = [
    ("Baseline", 0.1, 0.9, 0.2),
    ("alpha=0.5", 0.5, 0.9, 0.2),
    ("alpha=0.9", 0.9, 0.9, 0.2),
    ("gamma=0.5", 0.1, 0.5, 0.2),
    ("gamma=0.99", 0.1, 0.99, 0.2),
    ("epsilon=0.05", 0.1, 0.9, 0.05),
    ("epsilon=0.5", 0.1, 0.9, 0.5),
]

q_tables = {}
training_rewards = {}

for name, alpha, gamma, epsilon in tests:
    q, rewards = train(alpha, gamma, epsilon)
    q_tables[name] = q
    training_rewards[name] = rewards
    print(name, "finished")

### Make four training plots

We plot the **average reward over 1,000 games** instead of every individual reward. One Blackjack game can only finish with `-1`, `0`, or `+1`, so single-game rewards make a very noisy plot.

In [ ]:
def plot_results(names, title, filename):
    plt.figure(figsize=(8, 4))

    for name in names:
        rewards = training_rewards[name]
        average = np.convolve(rewards, np.ones(1000) / 1000, mode="valid")
        plt.plot(range(1000, EPISODES + 1), average, label=name)

    plt.title(title)
    plt.xlabel("Training episode")
    plt.ylabel("Average reward (last 1000 games)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("results/" + filename)
    plt.show()


plot_results(["Baseline"], "C: Baseline", "baseline.png")
plot_results(["Baseline", "alpha=0.5", "alpha=0.9"],
             "D: Learning rates", "alpha.png")
plot_results(["gamma=0.5", "Baseline", "gamma=0.99"],
             "E: Discount factors", "gamma.png")
plot_results(["epsilon=0.05", "Baseline", "epsilon=0.5"],
             "F: Exploration rates", "epsilon.png")

## G. Evaluate the best agent

Evaluation uses **no random exploration**. First, we choose the best setting using 1,000 validation games per agent. Then we compare the baseline and selected agent on **2,000 separate test games**. The assignment requires at least 30 evaluation episodes, so 2,000 is more than enough.

**Use the printed numbers for the report.** A mean reward closer to +1 is better. Since Blackjack has a house advantage, a negative mean reward can still be a reasonable result.

In [ ]:
def evaluate(q, games, seed):
    env = gym.make("Blackjack-v1", natural=False, sab=False)
    env.reset(seed=seed)
    scores = []

    for i in range(games):
        state, _ = env.reset()
        done = False
        while not done:
            action = best_action(q, state)  # No epsilon here.
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
        scores.append(reward)

    env.close()
    mean_reward = np.mean(scores)
    win_rate = 100 * scores.count(1.0) / games
    draw_rate = 100 * scores.count(0.0) / games
    return mean_reward, win_rate, draw_rate


validation = {}
for name in q_tables:
    validation[name] = evaluate(q_tables[name], 1000, 1234)[0]

best_name = max(validation, key=validation.get)
print("Best setting from validation:", best_name)

with open("results/evaluation.txt", "w") as file:
    file.write("Best setting: " + best_name + "\n")
    final_names = ["Baseline"]
    if best_name != "Baseline":
        final_names.append(best_name)
    else:
        print("The baseline was the best, so there is no separate better agent.")

    for name in final_names:
        mean, wins, draws = evaluate(q_tables[name], 2000, 2026)
        line = f"{name}: mean reward={mean:.4f}, wins={wins:.2f}%, draws={draws:.2f}%"
        print(line)
        file.write(line + "\n")

## Record the required MP4

This records five games of the **trained** best agent. The video is created from Gymnasium's `rgb_array` render mode, not from a screenshot or a simulated result.

In [ ]:
env = gym.make("Blackjack-v1", natural=False, sab=False, render_mode="rgb_array")
frames = []

for game in range(5):
    state, _ = env.reset(seed=500 + game)
    frames.append(env.render())
    done = False

    while not done:
        action = best_action(q_tables[best_name], state)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        frames.append(env.render())

env.close()
imageio.mimsave("results/blackjack_demo.mp4", frames, fps=2)
display(Video("results/blackjack_demo.mp4", embed=True))

## Download the results

Download the ZIP and keep the PNG plots, evaluation text, and MP4 together. Separately save this notebook with **File → Download → Download .ipynb** and upload it to GitHub with the results and README.

On Canvas, submit **only your PDF report**, containing a link to your **public** GitHub repository. Also include conclusions about how α, γ and ε affected learning. Do not invent results: use the numbers from your own run.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("blackjack_results", "zip", "results")
files.download("blackjack_results.zip")